# CDX index -> single-name constituents (Bloomberg BDS test)

Goal: deduce which single names are in each CDX index via `xbbg` BDS, then map to `market_cds_ticker` in `teams.structured_credit_abhutra.cds_transform_interpolated`.

Requires a running Bloomberg Terminal session (bbcomm) on this machine.

Note: if printing fails with a cp1252 UnicodeEncodeError, launch Jupyter with `PYTHONIOENCODING=utf-8`.

In [1]:
import sys
import pandas as pd
import xbbg

# own utils: not pip-installed, so add the repo root to sys.path
BBG_UTILS_PATH = 'S:/Structured Credit/Matt Stuff/claude_repos/bbg_utils'
if BBG_UTILS_PATH not in sys.path:
    sys.path.insert(0, BBG_UTILS_PATH)
from bbg_utils import bds_mm, bdp_mm, bdh_mm

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
print('xbbg', xbbg.__version__)

xbbg 1.5.1


## 1. Connectivity check

In [2]:
bdp_mm('SPX Index', 'PX_LAST')

,security,PX_LAST
0,SPX Index,7645.87


## 2. Pull constituents with BDS
`INDX_MEMBERS` works on the generic CDX tickers (verified: IG returns 125 rows incl. extra/defaulted lines, HY 100). Swap in a specific series (e.g. `CDX IG CDSI S45 5Y Corp`) for a historical roster.

In [3]:
INDEX_TICKERS = [
    'CDX IG CDSI GEN 5Y Corp',
    'CDX HY CDSI GEN 5Y Corp',
]
FIELD = 'INDX_MEMBERS'

# BDS columns come back generic; names below are inferred from the data, confirm them.
COLMAP = {
    'Member Ticker and Exchange Code': 'member_name',
    'column   1': 'coupon_or_weight',
    'column   2': 'currency',
    'column   3': 'seniority',
    'column   4': 'bbg_cds_ticker',
    'column   5': 'sp_id',
}

members = (bds_mm(INDEX_TICKERS, FIELD)
           .rename(columns=COLMAP)
           .rename(columns={'security': 'index_ticker'}))
print(members.groupby('index_ticker').size())
members.head(10)

index_ticker
CDX HY CDSI GEN 5Y Corp    100
CDX IG CDSI GEN 5Y Corp    125
dtype: int64


,index_ticker,member_name,coupon_or_weight,currency,seniority,bbg_cds_ticker,sp_id
0,CDX IG CDSI GEN 5Y Corp,Advanced Micro Devices Inc,0.8,USD,Senior,CT355268,SPEL0LCE
1,CDX IG CDSI GEN 5Y Corp,Honeywell International Inc,0.8,USD,Senior,CHON1U5,SPET0L8H
2,CDX IG CDSI GEN 5Y Corp,American Electric Power Co Inc,0.8,USD,Senior,CAEP1U5,SP2D0OQB
3,CDX IG CDSI GEN 5Y Corp,American Express Co,0.8,USD,Senior,CAXP1U5,SP9P0O23
4,CDX IG CDSI GEN 5Y Corp,American International Group I,0.8,USD,Senior,CAIG1U5,SPED0LO5
5,CDX IG CDSI GEN 5Y Corp,Arrow Electronics Inc,0.8,USD,Senior,CARW1U5,SP4L0LX7
6,CDX IG CDSI GEN 5Y Corp,AutoZone Inc,0.8,USD,Senior,CAZO1U5,SP9H0OGY
7,CDX IG CDSI GEN 5Y Corp,Avnet Inc,0.8,USD,Senior,CAVT1U5,SPF90L4V
8,CDX IG CDSI GEN 5Y Corp,Baxter International Inc,0.8,USD,Senior,CBAX1U5,SP190PGQ
9,CDX IG CDSI GEN 5Y Corp,Verizon Communications Inc,0.8,USD,Senior,CVZGF1U5,SPFH0KSU


## 3. Sanity checks
IG should be ~125 names and HY ~100. Check for duplicates and odd rows before mapping.

In [4]:
members.groupby('index_ticker').agg(n=('member_name','size'), n_unique=('member_name','nunique'))

,n,n_unique
index_ticker,,
CDX HY CDSI GEN 5Y Corp,100,100
CDX IG CDSI GEN 5Y Corp,125,125


In [5]:
# Map BBG constituents -> Market_CDS_Ticker via the mappings file.
# cds_mappings_final.csv: BBG_Short_Ticker == members.bbg_cds_ticker, CDS_Ticker == market_cds_ticker in Databricks.
MAPPINGS_PATH = 'S:/Structured Credit/Matt Stuff/cds_mappings/cds_mappings_final.csv'

mappings = (pd.read_csv(MAPPINGS_PATH, usecols=['CDS_Ticker', 'BBG_Short_Ticker'],
                        dtype=str, encoding='cp1252')   # file isn't utf-8
            .dropna(subset=['BBG_Short_Ticker'])
            .drop_duplicates('BBG_Short_Ticker')        # keys are unique today; guards a many-to-one blowup
            .rename(columns={'CDS_Ticker': 'Market_CDS_Ticker'}))

members = (members.drop(columns=['Market_CDS_Ticker'], errors='ignore')
           .merge(mappings, how='left', left_on='bbg_cds_ticker', right_on='BBG_Short_Ticker')
           .drop(columns='BBG_Short_Ticker'))

print(members.groupby('index_ticker')['Market_CDS_Ticker'].agg(n='size', mapped='count'))
print('Unmapped:')
display(members.loc[members['Market_CDS_Ticker'].isna(), ['index_ticker', 'member_name', 'bbg_cds_ticker']])
members.head(10)

                           n  mapped
index_ticker                        
CDX HY CDSI GEN 5Y Corp  100     100
CDX IG CDSI GEN 5Y Corp  125     125
Unmapped:


,index_ticker,member_name,bbg_cds_ticker


,index_ticker,member_name,coupon_or_weight,currency,seniority,bbg_cds_ticker,sp_id,Market_CDS_Ticker
0,CDX IG CDSI GEN 5Y Corp,Advanced Micro Devices Inc,0.8,USD,Senior,CT355268,SPEL0LCE,AMD
1,CDX IG CDSI GEN 5Y Corp,Honeywell International Inc,0.8,USD,Senior,CHON1U5,SPET0L8H,HON
2,CDX IG CDSI GEN 5Y Corp,American Electric Power Co Inc,0.8,USD,Senior,CAEP1U5,SP2D0OQB,AEP
3,CDX IG CDSI GEN 5Y Corp,American Express Co,0.8,USD,Senior,CAXP1U5,SP9P0O23,AXP
4,CDX IG CDSI GEN 5Y Corp,American International Group I,0.8,USD,Senior,CAIG1U5,SPED0LO5,AIG
5,CDX IG CDSI GEN 5Y Corp,Arrow Electronics Inc,0.8,USD,Senior,CARW1U5,SP4L0LX7,ARW
6,CDX IG CDSI GEN 5Y Corp,AutoZone Inc,0.8,USD,Senior,CAZO1U5,SP9H0OGY,AZO
7,CDX IG CDSI GEN 5Y Corp,Avnet Inc,0.8,USD,Senior,CAVT1U5,SPF90L4V,AVT
8,CDX IG CDSI GEN 5Y Corp,Baxter International Inc,0.8,USD,Senior,CBAX1U5,SP190PGQ,BAX
9,CDX IG CDSI GEN 5Y Corp,Verizon Communications Inc,0.8,USD,Senior,CVZGF1U5,SPFH0KSU,VZ


In [6]:
from cdx_members import get_index_members

ig = get_index_members('CDX IG CDSI GEN 5Y Corp')
print(len(ig), 'members;', ig['Market_CDS_Ticker'].isna().sum(), 'unmapped')
ig.head(10)

125 members; 0 unmapped


,index_ticker,member_name,coupon_or_weight,currency,seniority,bbg_cds_ticker,sp_id,Market_CDS_Ticker
0,CDX IG CDSI GEN 5Y Corp,Advanced Micro Devices Inc,0.8,USD,Senior,CT355268,SPEL0LCE,AMD
1,CDX IG CDSI GEN 5Y Corp,Honeywell International Inc,0.8,USD,Senior,CHON1U5,SPET0L8H,HON
2,CDX IG CDSI GEN 5Y Corp,American Electric Power Co Inc,0.8,USD,Senior,CAEP1U5,SP2D0OQB,AEP
3,CDX IG CDSI GEN 5Y Corp,American Express Co,0.8,USD,Senior,CAXP1U5,SP9P0O23,AXP
4,CDX IG CDSI GEN 5Y Corp,American International Group I,0.8,USD,Senior,CAIG1U5,SPED0LO5,AIG
5,CDX IG CDSI GEN 5Y Corp,Arrow Electronics Inc,0.8,USD,Senior,CARW1U5,SP4L0LX7,ARW
6,CDX IG CDSI GEN 5Y Corp,AutoZone Inc,0.8,USD,Senior,CAZO1U5,SP9H0OGY,AZO
7,CDX IG CDSI GEN 5Y Corp,Avnet Inc,0.8,USD,Senior,CAVT1U5,SPF90L4V,AVT
8,CDX IG CDSI GEN 5Y Corp,Baxter International Inc,0.8,USD,Senior,CBAX1U5,SP190PGQ,BAX
9,CDX IG CDSI GEN 5Y Corp,Verizon Communications Inc,0.8,USD,Senior,CVZGF1U5,SPFH0KSU,VZ


## 4. Next steps
- Identify which column maps to RED code / Markit ticker (`bbg_cds_ticker` vs `sp_id`).
- Join to `cds_transform_interpolated` on `red_code` or `markit_ticker`.
- Build the index -> names mapping table.